# Ingestion RAG d'un corpus long structure

*Recycler la documentation du projet LivresAgites (en sommeil) vers le depot
pedagogique public `jsboige/CoursIA`. Ce notebook enseigne la **methode** (comment
on decoupe un corpus long structure pour le RAG), jamais l'**instance** (le
catalogue d'une vraie maison d'edition). Le corpus ci-dessous est **synthetique**.*

[<- Parcours AI-Engine](README.md) | [Comparatif OWUI/AI-Engine](../comparatif-owui-vs-ai-engine.md)

## Le probleme

[`5_RAG_Modern.ipynb`](../../Texte/5_RAG_Modern.ipynb) montre comment decouper un
**texte continu unique** (le debat Lincoln-Douglas) en chunks. Mais un cas
d'usage editorial reel n'est pas un texte continu : c'est un **catalogue de
plusieurs ouvrages**, chacun structure en chapitres. Le defaut de chunking y est
different et plus subtil.

Ce notebook compare deux strategies d'ingestion d'un tel corpus :

1. **Chunking naif** (taille fixe + recouvrement) -- l'approche par defaut de
   beaucoup de tutoriels RAG.
2. **Chunking structure** (par chapitre, sous-decoupe si trop long) -- qui
   preserve la hierarchie du document source comme metadonnee de filtrage.

La these : **la degradation du retrieval vient du chunking, pas du modele
d'embeddings.** On le demontre avec un vectoriseur TF-IDF deterministe (pas de
cle d'API, pas de reseau) -- exactement pour isoler cette variable. Changez le
vectoriseur pour de vrais embeddings, la conclusion tient.

## 1. Configuration

Aucune cle d'API necessaire. Le vectoriseur TF-IDF de `scikit-learn` est
deterministe et s'execute hors ligne -- c'est ce qui rend le notebook
reproductible par un etudiant sans compte payant.

In [1]:
#dependances : scikit-learn (TF-IDF), numpy (cosinus). Aucun reseau.
import re
import textwrap
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Configuration OK -- vectoriseur TF-IDF deterministe (hors ligne).")

Configuration OK -- vectoriseur TF-IDF deterministe (hors ligne).


## 2. Le corpus : trois ouvrages structures en chapitres

Corpus synthetique de la **mediatheque de Valmont**. Trois ouvrages, chacun en
trois chapitres au theme nettement distinct. Les chapitres sont courts pour
rendre le defaut visible a l'oeil ; dans un cas reel, chaque chapitre ferait
des milliers de mots, mais la mecanique est identique.

L'important n'est pas le contenu -- c'est que **chaque chapitre traite d'un seul
theme**, de sorte que melanger deux chapitres dans un meme chunk produit un
fragment incoherent.

In [2]:
#Corpus synthetique (mediatheque de Valmont). Aucune prose reelle sous droits.
# Structure : chaque chapitre = (titre, texte). La paire (titre, texte) est ce
# que le chunking structure doit preserver comme etiquette. Les textes repetent
# les mots-theme du chapitre pour donner au vectoriseur un signal net.
CORPUS = {
    "Le Phare de Valmont": {
        "auteur": "A. Riviere (fictif)",
        "chapitres": {
            1: ("La tempete",
                "La tempete frappa la cote de Valmont une nuit de novembre. Le vent "
                "soufflait en rafales sur la digue, et les vagues battaient la pierre "
                "du phare avec une fureur reguliere. Malgre la tempete, le gardien "
                "alluma la lampe du phare a la tombee de la nuit, comme chaque soir. "
                "Sans cette lumiere, la tempete aurait pu jeter un navire sur les rochers."),
            2: ("Le gardien",
                "Me Lucien, gardien du phare depuis trente ans, connaissait chaque "
                "craquement de l'escalier en colimacon. Ce gardien tenait le journal "
                "de bord a jour, notant chaque navire passe, chaque tempete, chaque "
                "avarie de la lampe. Le role du gardien etait une veille patiente, "
                "nuit apres nuit, au sommet du phare."),
            3: ("Le sauvetage",
                "Au petit matin, une barque derivait vers les rochers apres la tempete. "
                "Le gardien appela les secours par la radio VHF : un sauvetage etait "
                "urgent. Le canot de sauvetage de la SNSM arriva en vingt minutes et "
                "le sauvetage du pecheur epuise reussit. Sans le gardien et son appel "
                "VHF, ce sauvetage aurait echoue."),
        },
    },
    "Jardins Suspendus": {
        "auteur": "M. Olivier (fictif)",
        "chapitres": {
            1: ("Les rosiers",
                "La roseraie du jardin suspendu comptait quarante varietes de rosiers. "
                "Le rosier ancien 'Rosa gallica' fleurissait une fois l'an, en juin. "
                "La taille des rosiers se faisait en fin d'hiver, avant la montee de "
                "seve. Un rosier mal taille perdait sa forme ; un rosier bien taille "
                "fleurissait abondamment dans le jardin."),
            2: ("L'irrigation",
                "L'irrigation du jardin reposait sur un goutte-a-goutte qui alimentait "
                "chaque pied en eau. Un programmateur reglait l'irrigation : trois "
                "cycles d'eau par jour en ete, un seul en hiver. Le recuperateur d'eau "
                "de pluie alimentait aussi l'irrigation et couvrait soixante pour cent "
                "des besoins en eau du jardin."),
            3: ("Les saisons",
                "Chaque saison dictait un travail au jardin. Au printemps, la taille de "
                "formation structurait les sujets ; en ete, l'arrosage et le paillage "
                "protegeaient du sec. L'automne etait la saison de la plantation ; "
                "l'hiver, la saison ou l'on taillait les rosiers non remontants. "
                "Ainsi le jardin suivait le rythme des saisons."),
        },
    },
    "Horlogerie Pratique": {
        "auteur": "P. Marchand (fictif)",
        "chapitres": {
            1: ("Le pendule",
                "La longueur du pendule regle la periode d'oscillation de l'horloge. "
                "Un pendule d'un metre bat la seconde. Pour faire avancer l'horloge, "
                "on raccourcit le pendule ; pour la faire retarder, on l'allonge. "
                "Le reglage du pendule est donc le reglage premier d'une horloge, "
                "et la masse du pendule n'a aucun effet sur la periode."),
            2: ("L'echappement",
                "L'echappement a ancre distribue l'energie de la roue motrice au "
                "balancier de l'horloge, une dent a la fois. C'est l'echappement qui "
                "regle le mouvement. Le lubrifiant des pivots de l'echappement "
                "s'epaissit avec le temps et ralentit l'horloge : il faut nettoyer "
                "l'echappement tous les cinq ans."),
            3: ("L'entretien",
                "L'entretien d'une horloge commence par le depoussierage au pinceau "
                "souple. Le boitier en laiton se polit au chiffon microfibre lors de "
                "l'entretien. On evite l'eau durant l'entretien, meme sur une horloge "
                "dite etanche, dont le joint doit etre controle chaque annee. Un bon "
                "entretien prolonge la vie de l'horloge."),
        },
    },
}

n_livres = len(CORPUS)
n_chapitres = sum(len(v["chapitres"]) for v in CORPUS.values())
print(f"Corpus : {n_livres} ouvrages, {n_chapitres} chapitres au total.")

Corpus : 3 ouvrages, 9 chapitres au total.


## 3. Deux strategies de chunking

On definit les deux strategies avant de les comparer. Le contraste est dans ce
que chaque chunk **preserve** comme metadonnee.

In [3]:
def chunk_naif(corpus, taille=180, recouvrement=40):
    # Chunking par taille fixe avec recouvrement. Ignore la structure.
    # Renvoie des chunks plats sans metadonnee de source ni de chapitre.
    texte_complet = "\n\n".join(
        para
        for livre in corpus.values()
        for ch in livre["chapitres"].values()
        for para in [ch[1]]
    )
    chunks = []
    debut = 0
    n = len(texte_complet)
    while debut < n:
        fin = min(debut + taille, n)
        chunks.append(texte_complet[debut:fin])
        if fin >= n:
            break
        debut = fin - recouvrement
    return chunks


def chunk_structure(corpus, taille_max=220):
    # Chunking par chapitre. Preserve livre + chapitre comme metadonnee.
    # Sous-decoupe un chapitre trop long, mais chaque chunk reste etiquete.
    chunks = []
    for titre, livre in corpus.items():
        auteur = livre["auteur"]
        for n_ch, (titre_ch, texte_ch) in livre["chapitres"].items():
            # si le chapitre depasse taille_max, on le decoupe par phrase
            morceaux = textwrap.wrap(texte_ch, width=taille_max, break_long_words=False)
            for morceau in morceaux:
                chunks.append({
                    "texte": morceau,
                    "livre": titre,
                    "auteur": auteur,
                    "chapitre": n_ch,
                    "titre_chapitre": titre_ch,
                })
    return chunks

print("Deux strategies definies : chunk_naif (taille fixe, sans etiquette) "
      "et chunk_structure (par chapitre, etiquette livre+chapitre).")

Deux strategies definies : chunk_naif (taille fixe, sans etiquette) et chunk_structure (par chapitre, etiquette livre+chapitre).


## 4. Application -- et le defaut apparait

In [4]:
naif = chunk_naif(CORPUS)
structure = chunk_structure(CORPUS)

print(f"Chunking naif      : {len(naif)} chunks, AUCUNE metadonnee de source.")
print(f"Chunking structure : {len(structure)} chunks, livre + chapitre preserves.")
print()
print("=== Un chunk naif au hasard (milieu du corpus) ===")
print(repr(naif[len(naif)//2]))
print()
print("=== Le chunk naif chevauche-t-il deux themes ? ===")
# on cherche un chunk naif qui contient des mots-cles de deux chapitres differents
marqueurs = {
    "tempete/phare": ["phare", "tempete", "gardien", "sauvetage"],
    "rosiers/jardin": ["rosier", "irrigation", "saisons", "jardin"],
    "horlogerie": ["pendule", "echappement", "horloge", "laiton"],
}
compteurs_chevauchement = 0
for chunk in naif:
    low = chunk.lower()
    themes = []
    for theme, mots in marqueurs.items():
        if any(m in low for m in mots):
            themes.append(theme)
    if len(themes) >= 2:
        compteurs_chevauchement += 1
print(f"Chunks naifs melangeant 2 themes (3 horizons) : {compteurs_chevauchement} sur {len(naif)}.")
print("-> Ces chunks melangent deux sujets et n'ont aucune etiquette de source.")

Chunking naif      : 20 chunks, AUCUNE metadonnee de source.
Chunking structure : 18 chunks, livre + chapitre preserves.

=== Un chunk naif au hasard (milieu du corpus) ===
"on : trois cycles d'eau par jour en ete, un seul en hiver. Le recuperateur d'eau de pluie alimentait aussi l'irrigation et couvrait soixante pour cent des besoins en eau du jardin."

=== Le chunk naif chevauche-t-il deux themes ? ===
Chunks naifs melangeant 2 themes (3 horizons) : 2 sur 20.
-> Ces chunks melangent deux sujets et n'ont aucune etiquette de source.


## 5. Vectorisation TF-IDF deterministe

On vectorise les chunks avec TF-IDF. Deterministe, hors ligne. Le modele
d'embeddings n'est PAS la variable de cette experience -- c'est le chunking.

In [5]:
#on vectorise les textes des deux strategies avec le MEME vectoriseur.
textes_naif = naif
textes_struct = [c["texte"] for c in structure]
corpus_total = textes_naif + textes_struct

vec = TfidfVectorizer(lowercase=True, sublinear_tf=True)
matrice = vec.fit_transform(corpus_total)

n_naif = len(textes_naif)
mat_naif = matrice[:n_naif]
mat_struct = matrice[n_naif:]
print(f"Matrice TF-IDF : {matrice.shape[0]} chunks x {matrice.shape[1]} termes.")

Matrice TF-IDF : 38 chunks x 246 termes.


## 6. Recherche -- meme requete, deux strategies

Cinq requetes ciblees, chacune portant sur un **theme de chapitre precis**.
Pour chaque requete, on regarde quel chunk remonte en tete avec chaque strategie.

La these predit : la strategie structuree renvoie le **bon chapitre du bon
livre** (coherence thematique + etiquette de source). La strategie naivee peut
renvoyer un fragment coherent (un bout de chapitre) **ou** un fragment qui
chevauche deux sujets.

In [6]:
REQUETES = [
    ("comment faire avancer ou retarder une horloge ?", "Horlogerie Pratique", 1),
    ("quand tailler les rosiers non remontants ?", "Jardins Suspendus", 3),
    ("comment le gardien a-t-il signale la barque ?", "Le Phare de Valmont", 3),
    ("a quoi sert l'echappement a ancre ?", "Horlogerie Pratique", 2),
    ("comment economiser l'eau au jardin ?", "Jardins Suspendus", 2),
]

def top_k(mat_chunks, chunks, req_vec, k=1):
    sims = cosine_similarity(req_vec, mat_chunks).ravel()
    idx = sims.argsort()[::-1][:k]
    return [(int(i), float(sims[i])) for i in idx]


def etiquetter_struct(c):
    return f"[{c['livre']} > ch.{c['chapitre']}]"


print(f"{'Requete':<46} | {'Attendu':<34} | Naif (top-1) | Structure (top-1)")
print("-" * 130)
exact_naif = 0
exact_struct = 0
for req, livre_attendu, ch_attendu in REQUETES:
    rv = vec.transform([req])
    # naif : on ne peut verifier le livre/chapitre (pas d'etiquette) -> on verifie la presence des mots-cles
    i_n, s_n = top_k(mat_naif, naif, rv)[0]
    bout_naif = naif[i_n][:40].replace("\n", " ")
    # structure : on a l'etiquette exacte
    i_s, s_s = top_k(mat_struct, structure, rv)[0]
    c_s = structure[i_s]
    ok_struct = (c_s["livre"] == livre_attendu and c_s["chapitre"] == ch_attendu)
    exact_struct += int(ok_struct)
    print(f"{req:<46} | {livre_attendu:<20} ch.{ch_attendu:<11} | s={s_n:.3f} {bout_naif!r}")
    print(f"{'':<46} | {'':<34} |              | {etiquetter_struct(c_s)} ok={ok_struct} s={s_s:.3f}")
    print()

print(f"Precision chapitre exact (top-1) : structure = {exact_struct}/{len(REQUETES)}, naif = N/A (pas d etiquette).")

Requete                                        | Attendu                            | Naif (top-1) | Structure (top-1)
----------------------------------------------------------------------------------------------------------------------------------
comment faire avancer ou retarder une horloge ? | Horlogerie Pratique  ch.1           | s=0.397 "orloge. Un pendule d'un metre bat la sec"
                                               |                                    |              | [Horlogerie Pratique > ch.1] ok=True s=0.412

quand tailler les rosiers non remontants ?     | Jardins Suspendus    ch.3           | s=0.380 "ujets ; en ete, l'arrosage et le paillag"
                                               |                                    |              | [Jardins Suspendus > ch.3] ok=True s=0.496

comment le gardien a-t-il signale la barque ?  | Le Phare de Valmont  ch.3           | s=0.262 'ampe. Le role du gardien etait une veill'
                                           

## 7. Lecture du resultat

La strategie structuree permet de **mesurer** la precision (on a l'etiquette
exacte livre + chapitre) : c'est l'avantage decisive. La strategie naivee, meme
quand elle remonte un fragment correct, est **invérifiable** -- l'utilisateur ne
sait pas de quel ouvrage ni de quel chapitre il vient, et un chunk a cheval sur
deux sujets est indiscernable d'un chunk coherent.

Ce qui est perdu avec le chunking naif :

- **l'etiquette de source** (quel ouvrage ?) -- impossible de filtrer par livre ;
- **l'etiquette de chapitre** (quelle partie ?) -- impossible de filtrer par section ;
- **la coherence thematique** -- un chunk qui chevauche deux chapitres peut
  contenir des affirmations contradictoires ou hors-contexte.

Ce qui est preserve avec le chunking structure :

- la **granularite** (chunk de taille raisonnable) -- on sous-decoupe si besoin ;
- la **traceabilite** (chaque chunk porte son origine) -- filtrable en payload ;
- la **coherence** (un chunk = un theme) -- le retrieval reste pertinent.

## 8. La pire chute : un chunk qui melange deux sujets

Demontrons qu'un chunk naif peut renvoyer une reponse **melangeant deux
chapitres**, ce qu'aucun chunk structure ne fera jamais par construction.

In [7]:
#Deterministe : on exhibe directement l'un des chunks naifs qui melangent 2 themes
# (la section 4 en a compte 2 sur 13). Aucune requete, aucun hasard : on montre
# un fragment hybride reellement produit par la decoupe a taille fixe.
hybrides = []
for chunk in naif:
    low = chunk.lower()
    themes = [t for t, mots in marqueurs.items() if any(m in low for m in mots)]
    if len(themes) >= 2:
        hybrides.append((themes, chunk))

print(f"Chunks naifs melangeant >= 2 themes : {len(hybrides)} sur {len(naif)}.")
if hybrides:
    themes, exemple = hybrides[0]
    print(f"Exemple -- themes melanges : {themes}")
    print(f"Contenu du chunk hybride :")
    print(f"  {exemple!r}")
    print()
    print("-> CE FRAGMENT EST INCOHERENT : il colle le debut d'un chapitre a la fin")
    print("   d'un autre. Servi comme contexte a un LLM, il melange deux sujets sans")
    print("   transition ni etiquette. Le LLM peut en tirer une reponse hybride, voire")
    print("   contradictoire. Et l'utilisateur ne sait pas d'ou vient le fragment.")
else:
    print("(Aucun chunk hybride pour cette taille de decoupe -- le defaut est")
    print(" probabiliste : faites varier 'taille' dans chunk_naif pour le voir apparaitre.)")
print()
print("Le chunking structure, lui, ne peut JAMAIS produire ce defaut : un chunk est")
print("entierement contenu dans un seul chapitre d'un seul ouvrage, par construction.")

Chunks naifs melangeant >= 2 themes : 2 sur 20.
Exemple -- themes melanges : ['tempete/phare', 'rosiers/jardin']
Contenu du chunk hybride :
  'vingt minutes et le sauvetage du pecheur epuise reussit. Sans le gardien et son appel VHF, ce sauvetage aurait echoue.\n\nLa roseraie du jardin suspendu comptait quarante varietes de'

-> CE FRAGMENT EST INCOHERENT : il colle le debut d'un chapitre a la fin
   d'un autre. Servi comme contexte a un LLM, il melange deux sujets sans
   transition ni etiquette. Le LLM peut en tirer une reponse hybride, voire
   contradictoire. Et l'utilisateur ne sait pas d'ou vient le fragment.

Le chunking structure, lui, ne peut JAMAIS produire ce defaut : un chunk est
entierement contenu dans un seul chapitre d'un seul ouvrage, par construction.


## 9. Ce qu'il faut retenir

> **Le chunking est le determinant cache du retrieval.** Un mauvais decoupage
> degrade la pertinence quelle que soit la qualite du modele d'embeddings --
> parce qu'un chunk qui melange deux sujets est un contexte incoherent, et qu'un
> chunk sans etiquette de source est un contexte invérifiable.

Regles operationnelles pour un corpus long structure :

1. **La structure du document guide le chunking**, pas une taille arbitraire.
   Un chapitre se decoupe par chapitre ; une section par section.
2. **Sous-decoupez les unites trop longues** plutot que de les fusionner --
   mais l'etiquette (livre, chapitre) voyage avec chaque sous-morceau.
3. **Indexez les etiquettes comme payload filtrable.** Permet "recherche dans
   l'ouvrage X seulement" -- impossible avec un chunking naif.
4. **Mesurez la precision** (requete -> bon chapitre ?) plutot que le recall seul.
   Un retrieval qui remonte le mauvais chapitre avec un score eleve est un
   defaut cache.

Ce notebook est volontairement isole de la question "quel modele d'embeddings ?"
(voir [`5_RAG_Modern.ipynb`](../../Texte/5_RAG_Modern.ipynb) et
[`01-Hands-On-Grounding.ipynb`](../../RAG-et-Memoire-Semantique/01-Hands-On-Grounding.ipynb)).
Ici, la variable est l'amont : **comment on prepare le corpus**.

## 10. Exercices

### Exercice 1 -- Mesurer l'effet du recouvrement

Le chunking naif prend un `recouvrement` en parametre. Implementez une fonction
qui mesure, pour plusieurs valeurs de `recouvrement` (0, 20, 40, 80), combien de
chunks naifs chevauchent deux themes (en reprenant la sonde `marqueurs` de la
section 4). Le recouvrement augmente-t-il ou diminue-t-il le defaut ? Pourquoi ?

In [8]:
# Exercice 1 -- effet du recouvrement sur le chevauchement de themes.
def compte_chevauchements(recouvrement):
    # TODO etudiant : chunk_naif(CORPUS, recouvrement=recouvrement), puis
    # compter les chunks melangeant >= 2 themes via la sonde marqueurs.
    # Renvoie (n_chunks, n_chevauchants).
    pass

# for r in [0, 20, 40, 80]:
#     n, bad = compte_chevauchements(r)
#     print(f"recouvrement={r:>3} : {bad}/{n} chunks hybrides")
print("Exercice 1 -- a implementer (decommenter la boucle ci-dessus).")

Exercice 1 -- a implementer (decommenter la boucle ci-dessus).


### Exercice 2 -- Etiqueter les chunks naifs a posteriori

On a obtenu des chunks naifs sans etiquette. Ecrire une fonction qui, pour un
chunk naif donne, tente de retrouver l'ouvrage et le chapitre d'origine par
recherche du texte dans le CORPUS. Quel pourcentage de chunks naifs sont
retrouvables ainsi ? (Indice : un chunk a cheval sur deux chapitres n'appartient
a aucun chapitre entier -- il est partiellement dans deux.) Conclusion sur la
robustesse de l'etiquetage a posteriori vs par construction.

In [9]:
# Exercice 2 -- etiquetage a posteriori des chunks naifs.
def retrouver_origine(chunk_texte, corpus):
    # TODO etudiant : pour chaque livre, chaque chapitre, verifier si
    # chunk_texte est SOUS-CHAINE du texte du chapitre. Renvoyer le couple
    # (livre, chapitre) ou None si non retrouve (cas du chevauchement).
    pass

# retrouves = sum(1 for c in naif if retrouver_origine(c, CORPUS) is not None)
# print(f"Chunks naifs etiquetables a posteriori : {retrouves}/{len(naif)}")
print("Exercice 2 -- a implementer.")

Exercice 2 -- a implementer.


## 11. Pour aller plus loin

- [`5_RAG_Modern.ipynb`](../../Texte/5_RAG_Modern.ipynb) -- les strategies de
  chunking sur un texte continu (le pendant mono-document).
- [`01-Hands-On-Grounding.ipynb`](../../RAG-et-Memoire-Semantique/01-Hands-On-Grounding.ipynb) --
  le retrieval Qdrant et le filtrage payload (le pendant infrastructure).
- [Comparatif OWUI/AI-Engine](../comparatif-owui-vs-ai-engine.md) -- ou cette
  ingestion se branche dans une plateforme GenAI.

---

*Note de transparence : ce notebook derive d'un cas d'usage reel (l'ingestion du
catalogue d'une maison d'edition dans une extension WordPress AI-Engine). Le
corpus, les titres, les autrices et tout contenu proviennent de la mediatheque
fictive de Valmont -- **aucune donnee client, aucun extrait de manuscrit sous
droits**. Ce qui remonte est la methode, jamais l'instance.*
